# Lab01 — Cotizaciones Óptimas de un Formador de Mercado

Este notebook **solo importa funciones de `src/` y genera gráficas**.
Toda la lógica de modelo y simulación vive en `src/model.py` y `src/simulation.py`.

In [1]:
import sys
import pathlib

import numpy as np

ROOT = pathlib.Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.model import (
    BASE_PI_I,
    BASE_PI_L,
    BASE_S0,
    expected_loss_ask,
    expected_loss_bid,
    optimize_quotes,
    price_pdf,
)
from src.simulation import monte_carlo, simulate_trades
from src.plots import (
    plot_loss_functions,
    plot_monte_carlo_totals,
    plot_pnl_distributions,
    plot_price_distribution,
)

SEED = 42
N_TRADES = 10_000
MC_RUNS = 1_000
MC_TRADES = 1_000

np.random.seed(SEED)

## 1. Distribución del precio verdadero $f(P)$

In [2]:
fig = plot_price_distribution(price_pdf, BASE_S0)
fig

<Figure size 880x550 with 1 Axes>

## 2. Optimización de Bid y Ask (caso base)

In [3]:
result = optimize_quotes(S0=BASE_S0, pi_I=BASE_PI_I, pi_L=BASE_PI_L)
print(f"Bid optimo:        {result['bid']:.2f}")
print(f"Ask optimo:        {result['ask']:.2f}")
print(f"Spread optimo:     {result['spread']:.2f}")
print(f"Utilidad esperada: {result['expected_utility']:.2f}")

Bid optimo:        16.45
Ask optimo:        23.43
Spread optimo:     6.98
Utilidad esperada: 0.84


## 3. Pérdida esperada frente a traders informados, por lado

In [4]:
A_range = np.linspace(BASE_S0, BASE_S0 + 10, 100)
B_range = np.linspace(0.5, BASE_S0, 100)
loss_ask = [expected_loss_ask(A) for A in A_range]
loss_bid = [expected_loss_bid(B) for B in B_range]

fig = plot_loss_functions(A_range, loss_ask, B_range, loss_bid)
fig

<Figure size 880x550 with 1 Axes>

## 4. Simulación de 10,000 trades bajo tres regímenes

In [5]:
regimes = {
    "Optimo": (result["bid"], result["ask"]),
    "Estrecho": (19.75, 20.05),
    "Amplio": (18.40, 21.40),
}

pnl_by_regime = {}
for name, (bid, ask) in regimes.items():
    pnl = simulate_trades(N_TRADES, bid, ask, BASE_S0, BASE_PI_I, BASE_PI_L)
    pnl_by_regime[name] = pnl
    print(f"{name:10s} Bid={bid:6.2f} Ask={ask:6.2f}  PnL total={pnl.sum():10.2f}  PnL medio={pnl.mean():7.4f}")

Optimo     Bid= 16.45 Ask= 23.43  PnL total=   8069.23  PnL medio= 0.8069
Estrecho   Bid= 19.75 Ask= 20.05  PnL total=  -7063.64  PnL medio=-0.7064
Amplio     Bid= 18.40 Ask= 21.40  PnL total=   3338.33  PnL medio= 0.3338


In [6]:
fig = plot_pnl_distributions(pnl_by_regime)
fig

<Figure size 880x550 with 1 Axes>

## 5. Monte Carlo: 1,000 corridas de 1,000 trades

In [7]:
totals_by_regime = {}
for name, (bid, ask) in regimes.items():
    totals = monte_carlo(MC_RUNS, MC_TRADES, bid, ask, BASE_S0, BASE_PI_I, BASE_PI_L)
    totals_by_regime[name] = totals
    print(f"{name:10s} media={totals.mean():10.2f}  std={totals.std():8.2f}")

Optimo     media=    841.59  std=   50.84
Estrecho   media=   -675.14  std=   43.70


Amplio     media=    327.38  std=   43.04


In [8]:
fig = plot_monte_carlo_totals(totals_by_regime)
fig

<Figure size 880x550 with 1 Axes>